## Generating LLM Fine-tuning Dataset Alpaca Format huggingface Dataset 

Please check `data/03_primary/llm_swow_finetune_dataset/README.md` for more information.

## Setup jupyter notebook lab from server 
one can setup the lab env by running the following command on local machine terminal: 

```bash
jupyter lab --no-browser --port=22549 -ServerApp.token='hello'
```

Then, on local machine terminal, run: 

```bash
ssh -N -f -L localhost:22549:localhost:22549 sukaih@spartan-gpgpu160
```

In [6]:
import jsonlines
import os 
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
from tqdm.auto import tqdm
from glob import glob
os.environ['WORKING_DIR']

'/data/gpfs/projects/punim2219/LM_with_SWOW/sukaih/cultural-lexis-finetune-llms'

In [7]:
swow_en_path = os.path.join(os.environ['WORKING_DIR'], 'data/01_raw/SWOW/swow.en.csv')

swow_zh_path = os.path.join(os.environ['WORKING_DIR'], 'data/01_raw/SWOW/SWOWZH.R55.20230424.processed.copy.xlsx')

swow_us_path = os.path.join(os.environ['WORKING_DIR'], 'data/01_raw/SWOW/SWOW-US.R100.20180827.processed.xlsx')

swow_nl_path = os.path.join(os.environ['WORKING_DIR'], 'data/01_raw/SWOW/SWOWNL.2012.processed.csv')

In [8]:
# en has no header so we need to add it as relation, cue, association and freq
cols = ['relation', 'cue', 'association', 'freq']
# swow_en_df = pd.read_csv(swow_en_path, names=cols, sep='\t')
# swow_zh_df = pd.read_excel(swow_zh_path)
# swow_us_df = pd.read_excel(swow_us_path)
swow_nl_df = pd.read_csv(swow_nl_path)

In [9]:
swow_nl_df
swow_focus_df = swow_nl_df
focus_name_tag = 'nl'

In [10]:
# remove rows where cue is NaN
print("Current shape of swow_focus_df:", swow_focus_df.shape)
swow_focus_df = swow_focus_df[~swow_focus_df['cue'].isna()]
print("Shape of swow_focus_df after removing NaN cues:", swow_focus_df.shape)

Current shape of swow_focus_df: (1496926, 4)
Shape of swow_focus_df after removing NaN cues: (1496926, 4)


In [11]:
# a function that groupby cue and sort association by freq

def group_sort(df):
    df = df.groupby('cue').apply(lambda x: x.sort_values('freq', ascending=False)).reset_index(drop=True)
    # get return a dictionary where the key is the cue and the value is the association list 
    get = df.groupby('cue')['association'].apply(list).to_dict()
    return get

In [12]:
# swow_en_dict = group_sort(swow_en_df)
# swow_zh_dict = group_sort(swow_zh_df)
# swow_us_dict = group_sort(swow_us_df)
swow_focus_dict = group_sort(swow_focus_df)

# print('length of en:', len(swow_en_dict))
# print('length of zh:', len(swow_zh_dict))
# print('length of us:', len(swow_us_dict))
print('length of focus:', len(swow_focus_dict))

/tmp/ipykernel_254204/13522512.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('cue').apply(lambda x: x.sort_values('freq', ascending=False)).reset_index(drop=True)


length of focus: 12571


In [13]:
def nested_group_sort(df):
    df = df.groupby('cue').apply(lambda x: x.sort_values('freq', ascending=False)).reset_index(drop=True)
    
    def create_nested_dict(group):
        return group.set_index('association')['freq'].to_dict()
    
    nested_dict = df.groupby('cue').apply(create_nested_dict).to_dict()
    return nested_dict

# swow_en_nested_dict = nested_group_sort(swow_en_df)
# swow_zh_nested_dict = nested_group_sort(swow_zh_df)
# swow_us_nested_dict = nested_group_sort(swow_us_df)
swow_focus_nested_dict = nested_group_sort(swow_focus_df)

/tmp/ipykernel_254204/4041144861.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('cue').apply(lambda x: x.sort_values('freq', ascending=False)).reset_index(drop=True)
/tmp/ipykernel_254204/4041144861.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  nested_dict = df.groupby('cue').apply(create_nested_dict).to_dict()


In [14]:
# save the dictionary
# nested_swow_en_path = os.path.join(os.environ['WORKING_DIR'], 'data/02_intermediate/nested_swow_en.pkl')
# nested_swow_zh_path = os.path.join(os.environ['WORKING_DIR'], 'data/02_intermediate/nested_swow_zh.pkl')
# nested_swow_us_path = os.path.join(os.environ['WORKING_DIR'], 'data/02_intermediate/nested_swow_us.pkl')
nested_swow_focus_path = os.path.join(os.environ['WORKING_DIR'], f'data/02_intermediate/nested_swow_{focus_name_tag}.pkl')

import pickle
# with open(nested_swow_en_path, 'wb') as f:
#     pickle.dump(swow_en_nested_dict, f)
    
# with open(nested_swow_zh_path, 'wb') as f:
#     pickle.dump(swow_zh_nested_dict, f)
    
# with open(nested_swow_us_path, 'wb') as f:
#     pickle.dump(swow_us_nested_dict, f)

with open(nested_swow_focus_path, 'wb') as f:
    pickle.dump(swow_focus_nested_dict, f)

In [15]:
# we want to expand the training length to 1M 
# ratio_en = 1000000 // len(swow_en_dict)

# ratio_zh = 1000000 // len(swow_zh_dict)

# ratio_us = 1000000 // len(swow_us_dict)

ratio_focus = 1000000 // len(swow_focus_dict)

train_validate_test_ratio = [0.7, 0.2 ,0.1]

def train_val_test_split(data_dict):
    # shuffle the data
    np.random.seed(42)
    keys = list(data_dict.keys())
    np.random.shuffle(keys)
    # split the data
    train_end = int(train_validate_test_ratio[0] * len(keys))
    val_end = int(train_validate_test_ratio[1] * len(keys)) + train_end
    train_keys = keys[:train_end]
    val_keys = keys[train_end:val_end]
    test_keys = keys[val_end:]
    # create the data
    train_data = {k: data_dict[k] for k in train_keys}
    val_data = {k: data_dict[k] for k in val_keys}
    test_data = {k: data_dict[k] for k in test_keys}
    return train_data, val_data, test_data

# swow_en_train, swow_en_val, swow_en_test = train_val_test_split(swow_en_dict)
# swow_zh_train, swow_zh_val, swow_zh_test = train_val_test_split(swow_zh_dict)
# swow_us_train, swow_us_val, swow_us_test = train_val_test_split(swow_us_dict)

swow_focus_train, swow_focus_val, swow_focus_test = train_val_test_split(swow_focus_dict)


def expand_train_data(train_data_dict, ratio):
    # the rule is we randomly sample 70% of the list while keep the ordering the same
    sample_ratio = 0.7
    expanded_train_data_lst = [] 
    for key, val_lst in tqdm(train_data_dict.items(), desc='Expanding training data'):
        cache_set = set()
        for _ in range(ratio):
            sample_size = int(sample_ratio * len(val_lst))
            if sample_size == 0:
                sample_size = 1
            sample_indices = np.random.choice(len(val_lst), sample_size, replace=False)
            # sort the indices
            sample_indices.sort()
            sample_lst = [val_lst[i] for i in sample_indices]
            tuple_sample = (key, sample_lst)
            hash_tuple = str(tuple_sample)
            if hash_tuple not in cache_set:
                cache_set.add(hash_tuple) # to avoid duplicate
                expanded_train_data_lst.append(tuple_sample)
            
    return expanded_train_data_lst # structure is [(key, [val_lst]), ...]
            

# expanded_en_train = expand_train_data(swow_en_train, ratio_en)
# expanded_zh_train = expand_train_data(swow_zh_train, ratio_zh)
# expanded_us_train = expand_train_data(swow_us_train, ratio_us)

expanded_focus_train = expand_train_data(swow_focus_train, ratio_focus)

Expanding training data:   0%|          | 0/8799 [00:00<?, ?it/s]

In [16]:
# print("Length of expanded en train:", len(expanded_en_train))
# print("Length of expanded zh train:", len(expanded_zh_train))
# print("Length of expanded us train:", len(expanded_us_train))
print("Length of expanded focus train:", len(expanded_focus_train))

# print("Length of en val:", len(swow_en_val))
# print("Length of zh val:", len(swow_zh_val))
# print("Length of us val:", len(swow_us_val))
print("Length of focus val:", len(swow_focus_val))

# print("Length of en test:", len(swow_en_test))
# print("Length of zh test:", len(swow_zh_test))
# print("Length of us test:", len(swow_us_test))
print("Length of focus test:", len(swow_focus_test))

Length of expanded focus train: 695121
Length of focus val: 2514
Length of focus test: 1258


In [17]:
def convert_dict_to_lst(data_dict):
    data_lst = []
    for key, val_lst in data_dict.items():
        data_lst.append((key, val_lst))
    return data_lst

# en_val = convert_dict_to_lst(swow_en_val)
# zh_val = convert_dict_to_lst(swow_zh_val)
# us_val = convert_dict_to_lst(swow_us_val)
focus_val = convert_dict_to_lst(swow_focus_val)

# en_test = convert_dict_to_lst(swow_en_test)
# zh_test = convert_dict_to_lst(swow_zh_test)
# us_test = convert_dict_to_lst(swow_us_test)
focus_test = convert_dict_to_lst(swow_focus_test)

In [18]:
# print("Length of expanded en train:", len(expanded_en_train))
# print("Length of expanded zh train:", len(expanded_zh_train))
# print("Length of expanded us train:", len(expanded_us_train))
print("Length of expanded focus train:", len(expanded_focus_train))

# print("Length of en val:", len(en_val))
# print("Length of zh val:", len(zh_val))
# print("Length of us val:", len(us_val))
print("Length of focus val:", len(focus_val))

# print("Length of en test:", len(en_test))
# print("Length of zh test:", len(zh_test))
# print("Length of us test:", len(us_test))
print("Length of focus test:", len(focus_test))

# shuffle 
from random import shuffle
# shuffle(expanded_en_train)
# shuffle(expanded_zh_train)
# shuffle(expanded_us_train)
shuffle(expanded_focus_train)
# shuffle(en_val)
# shuffle(zh_val)
# shuffle(us_val)
shuffle(focus_val)
# shuffle(en_test)
# shuffle(zh_test)
# shuffle(us_test)
shuffle(focus_test)


Length of expanded focus train: 695121
Length of focus val: 2514
Length of focus test: 1258


In [19]:
# temp save the data

# temp_save_data_dict = {
#     "en_train": expanded_en_train,
#     "zh_train": expanded_zh_train,
#     "en_val": en_val,
#     "zh_val": zh_val,
#     "en_test": en_test,
#     "zh_test": zh_test
# }

# temp_save_path = os.path.join(os.environ['WORKING_DIR'], 'data/01_raw/temp_save_data_dict.pkl')

# with open(temp_save_path, 'wb') as f:
#     pickle.dump(temp_save_data_dict, f)

In [20]:
en_alpaca_format = {
    "system": "You are a sophisticated language model designed to explore word associations comprehensively.",
    "instruction": "Given a cue word, your task is to generate a comprehensive list of words associated with the cue word. Aim to cover as many relevant contexts, uses, and meanings as possible without repeating similar concepts. List a target of {count_lower_bound} to {count_higher_bound} words that together provide a broad and insightful representation of all significant associations. Focus on revealing both common and unique aspects related to the cue word to ensure a balanced and thorough exploration of potential associations. Words should be distinct from each other. Your response shall only be the list of associated words. Do not generate words conditioned on the presence of other words but rather focus on the cue word itself.",
    "input": "cue word: {cue_word}",
    "output": "{association_words}"
}

zh_alpaca_format = {
    "system": "您是一款专为全面探索词语关联而设计的高级语言模型。",
    "instruction": "给定一个提示词，你的任务是生成一个与该提示词相关联的全面词汇列表。目标是尽可能涵盖所有相关的语境、用法和含义，避免重复相似的概念。列出目标数量为 {count_lower_bound} 到 {count_higher_bound} 个词，这些词共同提供对所有重要关联的广泛而深刻的表示。专注于揭示与提示词相关的常见和独特的方面，以确保对潜在关联进行平衡而彻底的探索。词语应彼此不同。你的回答只能是相关联的词语列表。不要生成受其他词语存在影响的词语，而是专注于提示词本身。",
    "input": "提示词：{cue_word}",
    "output": "{association_words}"
}

# nl is Dutch
nl_alpaca_format = {
    "system": "Je bent een geavanceerd taalmodel dat is ontworpen om woordassociaties op een uitgebreide manier te verkennen.",
    "instruction": "Gegeven een sleutelwoord, is het jouw taak om een uitgebreide lijst van woorden te genereren die geassocieerd zijn met het sleutelwoord. Streef ernaar om zoveel mogelijk relevante contexten, gebruikswijzen en betekenissen te dekken zonder vergelijkbare concepten te herhalen. Maak een lijst met als doel {count_lower_bound} tot {count_higher_bound} woorden die samen een breed en inzichtelijk beeld geven van alle belangrijke associaties. Richt je op het onthullen van zowel algemene als unieke aspecten die verband houden met het sleutelwoord om zo een evenwichtige en grondige verkenning van potentiële associaties te waarborgen. Woorden moeten van elkaar onderscheiden zijn. Je reactie mag alleen de lijst met geassocieerde woorden bevatten. Genereer geen woorden die afhankelijk zijn van de aanwezigheid van andere woorden, maar richt je op het sleutelwoord zelf.",
    "input": "sleutelwoord: {cue_word}",
    "output": "{association_words}"
}

def convert_to_zh_alpaca_format(data_lst):
    alpaca_data_lst = []
    for key, val_lst in tqdm(data_lst, desc='Converting to zh alpaca format'):
        len_val_lst = len(val_lst)
        # get a lower bound and higher bound within range of +- 10% of the length, with noise being 3% of the length
        noise = int(0.03 * len_val_lst)
        # noise sign 
        noise_sign = np.random.choice([-1, 1])
        lower_bound = len_val_lst - int(0.1 * len_val_lst) + noise_sign * noise
        higher_bound = len_val_lst + int(0.1 * len_val_lst) + noise_sign * noise
        lower_bound = max(1, lower_bound)
        higher_bound = max(1, higher_bound)
        lower_bound = int(lower_bound)
        higher_bound = int(higher_bound)
        # remove nan in val_lst
        val_lst = [val for val in val_lst if val == val]
        
        data = {
            "instruction": zh_alpaca_format['instruction'].format(count_lower_bound=lower_bound, count_higher_bound=higher_bound),
            "input": key,
            "output": "， ".join(val_lst),
            "system": zh_alpaca_format['system']
        }
        alpaca_data_lst.append(data)
    return alpaca_data_lst

def convert_to_en_alpaca_format(data_lst):
    alpaca_data_lst = []
    for key, val_lst in tqdm(data_lst, desc='Converting to en alpaca format'):
        len_val_lst = len(val_lst)
        # get a lower bound and higher bound within range of +- 10% of the length, with noise being 3% of the length
        noise = int(0.03 * len_val_lst)
        # noise sign 
        noise_sign = np.random.choice([-1, 1])
        lower_bound = len_val_lst - int(0.1 * len_val_lst) + noise_sign * noise
        higher_bound = len_val_lst + int(0.1 * len_val_lst) + noise_sign * noise
        lower_bound = max(1, lower_bound)
        higher_bound = max(1, higher_bound)
        lower_bound = int(lower_bound)
        higher_bound = int(higher_bound)
        # remove nan in val_lst
        val_lst = [val for val in val_lst if val == val]
        
        data = {
            "instruction": en_alpaca_format['instruction'].format(count_lower_bound=lower_bound, count_higher_bound=higher_bound),
            "input": key,
            "output": ", ".join(val_lst),
            "system": en_alpaca_format['system']
        }
        alpaca_data_lst.append(data)
    return alpaca_data_lst
        
def convert_to_general_alpaca_format(data_lst, alpaca_format):
    alpaca_data_lst = []
    for key, val_lst in tqdm(data_lst, desc='Converting to general alpaca format'):
        len_val_lst = len(val_lst)
        # get a lower bound and higher bound within range of +- 10% of the length, with noise being 3% of the length
        noise = int(0.03 * len_val_lst)
        # noise sign 
        noise_sign = np.random.choice([-1, 1])
        lower_bound = len_val_lst - int(0.1 * len_val_lst) + noise_sign * noise
        higher_bound = len_val_lst + int(0.1 * len_val_lst) + noise_sign * noise
        lower_bound = max(1, lower_bound)
        higher_bound = max(1, higher_bound)
        lower_bound = int(lower_bound)
        higher_bound = int(higher_bound)
        # remove nan in val_lst
        val_lst = [val for val in val_lst if val == val]
        
        data = {
            "instruction": alpaca_format['instruction'].format(count_lower_bound=lower_bound, count_higher_bound=higher_bound),
            "input": key,
            "output": ", ".join(val_lst),
            "system": alpaca_format['system']
        }
        alpaca_data_lst.append(data)
    return alpaca_data_lst
        
        
# do the conversion
# en
print('Converting to alpaca format')
# train 
print('Train')
# en_train_alpaca_data = convert_to_en_alpaca_format(expanded_en_train)
# us_train_alpaca_data = convert_to_en_alpaca_format(expanded_us_train)
focus_alpaca_format = globals()[f"{focus_name_tag}_alpaca_format"]
focus_train_alpaca_data = convert_to_general_alpaca_format(expanded_focus_train, focus_alpaca_format)
# val
print('Val')
# en_val_alpaca_data = convert_to_en_alpaca_format(en_val)
# us_val_alpaca_data = convert_to_en_alpaca_format(us_val)
focus_val_alpaca_data = convert_to_general_alpaca_format(focus_val, focus_alpaca_format)
# test
print('Test')
# en_test_alpaca_data = convert_to_en_alpaca_format(en_test)
# us_test_alpaca_data = convert_to_en_alpaca_format(us_test)
focus_test_alpaca_data = convert_to_general_alpaca_format(focus_test, focus_alpaca_format)
        
# zh
# print('Converting to ZH alpaca format')
# # train
# print('Train')
# zh_train_alpaca_data = convert_to_zh_alpaca_format(expanded_zh_train)
# # val
# print('Val')
# zh_val_alpaca_data = convert_to_zh_alpaca_format(zh_val)
# # test
# print('Test')
# zh_test_alpaca_data = convert_to_zh_alpaca_format(zh_test)

        

Converting to alpaca format
Train


Converting to general alpaca format:   0%|          | 0/695121 [00:00<?, ?it/s]

Val


Converting to general alpaca format:   0%|          | 0/2514 [00:00<?, ?it/s]

Test


Converting to general alpaca format:   0%|          | 0/1258 [00:00<?, ?it/s]

In [21]:
entry_limit = 8000

# we now save the data into jsonl format 

def save_data_to_jsonl(data_lst, save_path):
    count = 0
    fp_offset = 0
    pbar = tqdm(total=len(data_lst), desc=f'Saving data for {save_path[-15:]}')
    while count < len(data_lst):
        save_fp = save_path.replace('.jsonl', f'_{fp_offset}.jsonl')
        with jsonlines.open(save_fp, mode='w') as writer:
            for data in data_lst[count:count+entry_limit]:
                writer.write(data)
                
        count += entry_limit
        pbar.update(entry_limit)
        
        fp_offset += 1
        
        
# # en
# print('Saving EN data')
# # train
# print('Train')
# train_en_fp = os.path.join(os.environ['WORKING_DIR'], 'data/03_primary/llm_swow_finetune_dataset/swow_en/train/chunk.jsonl')
# Path(os.path.dirname(train_en_fp)).mkdir(parents=True, exist_ok=True)
# save_data_to_jsonl(en_train_alpaca_data, train_en_fp)

# # val
# print('Val')
# val_en_fp = os.path.join(os.environ['WORKING_DIR'], 'data/03_primary/llm_swow_finetune_dataset/swow_en/trl/chunk.jsonl')
# Path(os.path.dirname(val_en_fp)).mkdir(parents=True, exist_ok=True)
# save_data_to_jsonl(en_val_alpaca_data, val_en_fp)

# # test
# print('Test')
# test_en_fp = os.path.join(os.environ['WORKING_DIR'], 'data/03_primary/llm_swow_finetune_dataset/swow_en/test/chunk.jsonl')
# Path(os.path.dirname(test_en_fp)).mkdir(parents=True, exist_ok=True)
# save_data_to_jsonl(en_test_alpaca_data, test_en_fp)


# us
# print('Saving US data')
# # train
# print('Train')
# train_us_fp = os.path.join(os.environ['WORKING_DIR'], 'data/03_primary/llm_swow_finetune_dataset/swow_us/train/chunk.jsonl')
# Path(os.path.dirname(train_us_fp)).mkdir(parents=True, exist_ok=True)
# save_data_to_jsonl(us_train_alpaca_data, train_us_fp)

# # val
# print('Val')
# val_us_fp = os.path.join(os.environ['WORKING_DIR'], 'data/03_primary/llm_swow_finetune_dataset/swow_us/trl/chunk.jsonl')
# Path(os.path.dirname(val_us_fp)).mkdir(parents=True, exist_ok=True)
# save_data_to_jsonl(us_val_alpaca_data, val_us_fp)

# # test
# print('Test')
# test_us_fp = os.path.join(os.environ['WORKING_DIR'], 'data/03_primary/llm_swow_finetune_dataset/swow_us/test/chunk.jsonl')
# Path(os.path.dirname(test_us_fp)).mkdir(parents=True, exist_ok=True)
# save_data_to_jsonl(us_test_alpaca_data, test_us_fp)


# # zh
# print('Saving ZH data')
# # train
# print('Train')
# train_zh_fp = os.path.join(os.environ['WORKING_DIR'], 'data/03_primary/llm_swow_finetune_dataset/swow_zh/train/chunk.jsonl')
# Path(os.path.dirname(train_zh_fp)).mkdir(parents=True, exist_ok=True)
# save_data_to_jsonl(zh_train_alpaca_data, train_zh_fp)

# # val
# print('Val')
# val_zh_fp = os.path.join(os.environ['WORKING_DIR'], 'data/03_primary/llm_swow_finetune_dataset/swow_zh/trl/chunk.jsonl')
# Path(os.path.dirname(val_zh_fp)).mkdir(parents=True, exist_ok=True)
# save_data_to_jsonl(zh_val_alpaca_data, val_zh_fp)

# # test
# print('Test')
# test_zh_fp = os.path.join(os.environ['WORKING_DIR'], 'data/03_primary/llm_swow_finetune_dataset/swow_zh/test/chunk.jsonl')
# Path(os.path.dirname(test_zh_fp)).mkdir(parents=True, exist_ok=True)
# save_data_to_jsonl(zh_test_alpaca_data, test_zh_fp)


# focus
print(f'Saving {focus_name_tag.upper()} data')
# train
print('Train')
train_focus_fp = os.path.join(os.environ['WORKING_DIR'], f'data/03_primary/llm_swow_finetune_dataset/swow_{focus_name_tag}/train/chunk.jsonl')
Path(os.path.dirname(train_focus_fp)).mkdir(parents=True, exist_ok=True)
save_data_to_jsonl(focus_train_alpaca_data, train_focus_fp)

# val
print('Val')
val_focus_fp = os.path.join(os.environ['WORKING_DIR'], f'data/03_primary/llm_swow_finetune_dataset/swow_{focus_name_tag}/trl/chunk.jsonl')
Path(os.path.dirname(val_focus_fp)).mkdir(parents=True, exist_ok=True)
save_data_to_jsonl(focus_val_alpaca_data, val_focus_fp)

# test
print('Test')
test_focus_fp = os.path.join(os.environ['WORKING_DIR'], f'data/03_primary/llm_swow_finetune_dataset/swow_{focus_name_tag}/test/chunk.jsonl')
Path(os.path.dirname(test_focus_fp)).mkdir(parents=True, exist_ok=True)
save_data_to_jsonl(focus_test_alpaca_data, test_focus_fp)



Saving NL data
Train


Saving data for ain/chunk.jsonl:   0%|          | 0/695121 [00:00<?, ?it/s]

Val


Saving data for trl/chunk.jsonl:   0%|          | 0/2514 [00:00<?, ?it/s]

Test


Saving data for est/chunk.jsonl:   0%|          | 0/1258 [00:00<?, ?it/s]

## Test the datasets library whether they can be loaded correctly



In [22]:
import datasets

In [23]:
print(os.environ.get('HF_HOME'))

/data/projects/punim0478/sukaih/huggingface


In [25]:
# RMB remove cache first 

dataset_path = os.path.join(os.environ['WORKING_DIR'], 'data/03_primary/llm_swow_finetune_dataset')

sample_data_config = "swow_nl"

sample_split = "train"

sample_dataset = datasets.load_dataset(
    dataset_path, 
    sample_data_config,
    split=sample_split
)

Resolving data files:   0%|          | 0/87 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/87 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating trl split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating trl_ranking split: 0 examples [00:00, ? examples/s]

In [26]:
sample_dataset[5]

{'instruction': 'Gegeven een sleutelwoord, is het jouw taak om een uitgebreide lijst van woorden te genereren die geassocieerd zijn met het sleutelwoord. Streef ernaar om zoveel mogelijk relevante contexten, gebruikswijzen en betekenissen te dekken zonder vergelijkbare concepten te herhalen. Maak een lijst met als doel 70 tot 86 woorden die samen een breed en inzichtelijk beeld geven van alle belangrijke associaties. Richt je op het onthullen van zowel algemene als unieke aspecten die verband houden met het sleutelwoord om zo een evenwichtige en grondige verkenning van potentiële associaties te waarborgen. Woorden moeten van elkaar onderscheiden zijn. Je reactie mag alleen de lijst met geassocieerde woorden bevatten. Genereer geen woorden die afhankelijk zijn van de aanwezigheid van andere woorden, maar richt je op het sleutelwoord zelf.',
 'input': 'kadetje',
 'output': 'x, leger, pistolet, opel, auto, voetbal, kleintje, matroos, deugniet, soldaatje, kereltje, soldaat, sport, sloeber,

In [19]:
# import llama3 tokenizer and find the max length of for the cutoff_len 
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
tokenizer.pad_token = tokenizer.eos_token

def find_max_length(dataset, tokenizer):
    # max_len = 0
    # for data in tqdm(dataset, desc='Finding max length'):
    #     input_text = data['instruction'] + ' ' + data['input'] + ' ' + data['output']
    #     input_len = len(tokenizer(input_text)['input_ids'])
    #     input_len = 2 * input_len # for the padding
    #     max_len = max(max_len, input_len)
        
    # alternatively, do batch encoding
    input_texts = [data['instruction'] + ' ' + data['input'] + ' ' + data['output'] for data in dataset]
    
    split_size = 1000
    
    input_texts = [input_texts[i:i+split_size] for i in range(0, len(input_texts), split_size)]
    
    length_lst = []
    
    for i in tqdm(range(len(input_texts)), desc='Finding max length'):
        input_lens = tokenizer(input_texts[i], padding=True, truncation=True, return_length=True)['length']
        length_lst.extend(input_lens)
        
    
    return length_lst
    
    

length_lst = find_max_length(sample_dataset, tokenizer)

Finding max length:   0%|          | 0/696 [00:00<?, ?it/s]

In [20]:
max(length_lst)
# 510 for swow_zh
# 610 for swow_en


625

In [21]:
tokenizer.model_max_length

131072